> **Reconstrucción de la v1** de `08_etl_transformaciones_kpis`. **No es el archivo original** (no se conserva): está deducida de v2 quitando lo que se añadió después, es decir, las secciones de tipo de problema (Días 5-6), feature engineering (Día 8) y serialización con joblib (Día 11).
> Contiene lo esencial de la Entrega 2: T1–T5 desde los CSV, 6 KPIs con histórico y verificación. Sustituida por v4 (versión final).

# 08 — ETL: Transformaciones y KPIs
### TFM · Análisis multidimensional del estrés estudiantil · Entrega 2


---
## Estructura de este notebook

| Sección | Contenido |
|---|---|
| Paso 0 | Configuración y conexión MySQL |
| T1-T5 | Cinco transformaciones ETL → tablas `dwh_*` |
| KPIs | Seis indicadores con histórico de ejecuciones |
| Verificación | Lectura de KPIs desde la base |

**Requisitos previos:** XAMPP con MySQL en verde; `modelo_datos_mysql_v2.sql` y `esquema_estrella_powerbi.sql` importados; `modelo_dwh_mysql.sql` importado (crea las tablas DWH vacías); notebook `07_carga_datos_mysql.ipynb` ejecutado.

## Paso 0 — Configuración y conexión

In [ ]:
import os, json
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
from sqlalchemy import create_engine, text

# ── Rutas ──────────────────────────────────────────────────────────────────
BASE = r"C:\Users\nicop\OneDrive\Documentos\Maestria\INESDI & UNIE\BA & IA\TFM\Propuesta"
ALMC = r"C:\Users\nicop\OneDrive\Documentos\Maestria\INESDI & UNIE\BA & IA\TFM\Propuesta\Entrega\Objetivo2"
FIG  = os.path.join(ALMC, "figuras_etl")
os.makedirs(FIG, exist_ok=True)

# ── Conexión MySQL (XAMPP/MariaDB) ─────────────────────────────────────────
DB_USER, DB_PASS = "root", ""
DB_HOST, DB_NAME = "localhost", "tfm_estres_estudiantil_2"
engine = create_engine(f"mysql+pymysql://{DB_USER}:{DB_PASS}@{DB_HOST}/{DB_NAME}?charset=utf8mb4")

EJECUCION = datetime.now()
print("Conectado a", DB_NAME)
print("Ejecución:", EJECUCION.strftime("%Y-%m-%d %H:%M:%S"))

---
## Transformaciones ETL → Datawarehouse

Las siguientes cinco transformaciones generan las tablas `dwh_*` del datawarehouse. **Requisito:** haber ejecutado `modelo_dwh_mysql.sql` en phpMyAdmin antes de correr estas celdas.

In [ ]:
# ── Funciones de armonización (mismas que notebook 06/07) ──────────────────
def map_genero(v):
    if pd.isna(v): return "NR"
    s = str(v).strip().lower()
    if s in ("male","m","hombre","1","1.0"):  return "M"
    if s in ("female","f","mujer","2","2.0"): return "F"
    if s in ("","nan","prefer not to say"):   return "NR"
    return "O"

def map_rango_edad(v):
    try: e = float(v)
    except: return "NR"
    if pd.isna(e) or e<13 or e>90: return "NR"
    if e<18: return "<18"
    if e<=25: return "18-25"
    if e<=35: return "26-35"
    if e<=50: return "36-50"
    return ">50"

def likert(v):
    try: x=float(v)
    except: return "No aplica"
    return "Bajo" if x<=2 else "Moderado" if x<=3 else "Alto"

def gad(v):
    try: x=float(v)
    except: return "No aplica"
    return "Bajo" if x<=4 else "Moderado" if x<=9 else "Alto"

def dass(v): return "Bajo" if v<=28 else "Moderado" if v<=39 else "Alto"

GEN_MAP = {"M":"Masculino","F":"Femenino","O":"Otro","NR":"No reportado"}
ID_GEN  = {"Masculino":1,"Femenino":2,"Otro":3,"No reportado":0}
ID_RNG  = {"No reportado":0,"NR":0,"<18":1,"18-25":2,"26-35":3,"36-50":4,">50":5}
ID_EDU  = {"No reportado":0,"Secundaria o menos":1,"Grado universitario":2,"Posgrado":3}
ID_EST  = {"No aplica":0,"Bajo":1,"Moderado":2,"Alto":3}
ID_DS   = {"B1_ACADSTRESS":1,"B1_ANXIETY":2,"B1_MHSURVEY":3,"B2_SMMH":6,
           "B3_DREADDIT_TR":8,"B3_DREADDIT_TE":9,"B3_DEPSTUDENT":10,"B4_DASS":11,"B4_MENTALHEALTH":12}
NR = lambda n: ["NR"]*n
EDUNR = lambda n: ["No reportado"]*n
print("Funciones listas.")

### T1 — Hechos unificados (integración de 9 fuentes reales)

`dwh_hechos_estres_unificado` — Une las 9 fuentes con señal real (uso_modelado='incluido') en una tabla de hechos con claves conformes. Excluye los 3 datasets sintéticos conforme a la política acordada.

In [ ]:
registros = []
def agregar(codigo, bloque, serie_gen, serie_edad, serie_edu, serie_estres):
    gen = [map_genero(x) for x in serie_gen]
    for (gcod, glbl), ed, edu, es in zip(
            [(g, GEN_MAP[g]) for g in gen], serie_edad, serie_edu, serie_estres):
        registros.append({"codigo_dataset":codigo,"bloque":bloque,
            "genero":glbl,"rango_edad":ed,"nivel_educativo":edu,"nivel_estres":es,
            "demografia_completa":(gcod!="NR" and ed!="NR")})

d=pd.read_csv(os.path.join(BASE,"Bloque 1","1 Student Academic Stress Real World Dataset","academic Stress level - maintainance 1.csv")); d.columns=[c.strip() for c in d.columns]
edu=d["Your Academic Stage"].map({"undergraduate":"Grado universitario","post-graduate":"Posgrado","high school":"Secundaria o menos"}).fillna("No reportado")
agregar("B1_ACADSTRESS","Factores internos",NR(len(d)),NR(len(d)),edu,d["Rate your academic stress index"].map(likert))

d=pd.read_csv(os.path.join(BASE,"Bloque 1","1 Student anxiety dataset","anxiety.csv"),encoding="latin1")
agregar("B1_ANXIETY","Factores internos",d["Gender"],d["Age"].map(map_rango_edad),EDUNR(len(d)),d["GAD_T"].map(gad))

d=pd.read_csv(os.path.join(BASE,"Bloque 1","1 Student Mental Health Survey","MentalHealthSurvey.csv")); d.columns=[c.strip() for c in d.columns]
edu=d["degree_level"].map({"Undergraduate":"Grado universitario","Postgraduate":"Posgrado"}).fillna("No reportado")
agregar("B1_MHSURVEY","Factores internos",d["gender"],d["age"].map(map_rango_edad),edu,d["depression"].map(likert))

d=pd.read_csv(os.path.join(BASE,"Bloque 2","2 Social Media and Mental Health","smmh.csv")); c=list(d.columns)
agregar("B2_SMMH","Habitos digitales",d[c[2]],d[c[1]].map(map_rango_edad),EDUNR(len(d)),pd.to_numeric(d[c[18]],errors="coerce").map(likert))

for part in ["train","test"]:
    d=pd.read_csv(os.path.join(BASE,"Bloque 3","3 Stress Analysis in Social Media Dataset",f"dreaddit-{part}.csv"))
    d=d[~d["text"].str.strip().isin(["#NAME?","#REF!","#VALUE!"])]
    agregar(f"B3_DREADDIT_{part[:2].upper()}","Percepcion emocional",NR(len(d)),NR(len(d)),EDUNR(len(d)),d["label"].map({0:"Bajo",1:"Alto"}))

d=pd.read_csv(os.path.join(BASE,"Bloque 3","3 Student_Mental_Health_Analysis GITHUB","Depression+Student+Dataset.csv"))
agregar("B3_DEPSTUDENT","Percepcion emocional",d["Gender"],d["Age"].map(map_rango_edad),EDUNR(len(d)),d["Depression"].map({"No":"Bajo","Yes":"Alto"}))

d=pd.read_csv(os.path.join(BASE,"Bloque 4","4 Depression Anxiety Stress Scales Responses","data.csv"),sep="\t",low_memory=False)
d=d[(d["age"]>=13)&(d["age"]<=90)].copy()
stri=[1,6,8,11,12,14,18,22,27,29,32,33,35,39]; d["strs"]=d[[f"Q{i}A" for i in stri]].sum(axis=1)
edu=d["education"].map({1:"Secundaria o menos",2:"Secundaria o menos",3:"Grado universitario",4:"Posgrado"}).fillna("No reportado")
agregar("B4_DASS","Validacion psicologica",d["gender"],d["age"].map(map_rango_edad),edu,d["strs"].map(dass))

d=pd.read_csv(os.path.join(BASE,"Bloque 4","4 Mental Health Dataset","Mental Health dataset1.csv"))
agregar("B4_MENTALHEALTH","Validacion psicologica",d["Gender"],NR(len(d)),EDUNR(len(d)),d["IncreasingStress"].map({"No":"Bajo","Maybe":"Moderado","Yes":"Alto"}).fillna("No aplica"))

T1=pd.DataFrame(registros)
T1["id_dataset"]=T1["codigo_dataset"].map(ID_DS)
T1["id_genero"]=T1["genero"].map(ID_GEN)
T1["id_rango_edad"]=T1["rango_edad"].map(ID_RNG)
T1["id_nivel_educativo"]=T1["nivel_educativo"].map(ID_EDU)
T1["id_nivel_estres"]=T1["nivel_estres"].map(ID_EST)
T1=T1[["id_dataset","codigo_dataset","bloque","id_genero","genero","id_rango_edad","rango_edad","id_nivel_educativo","nivel_educativo","id_nivel_estres","nivel_estres","demografia_completa"]]
T1.to_sql("dwh_hechos_estres_unificado",engine,if_exists="append",index=False,chunksize=5000)
print(f"T1: {len(T1):,} registros → dwh_hechos_estres_unificado")
print(T1["bloque"].value_counts().to_string())

**Opinión.** La unificación confirma que la Validación Psicológica domina en volumen (94%). Para los dashboards del Objetivo 3 esto es ventaja (base estadística robusta); para el modelado del Objetivo 2 se trabajará dataset por dataset, manteniendo el enfoque de análisis en paralelo.

### T2 — Agregado de prevalencia (agregación)

In [ ]:
T2=T1.groupby(["bloque","genero","rango_edad"]).agg(
    n_total=("nivel_estres","size"),
    n_estres_alto=("nivel_estres",lambda s:(s=="Alto").sum()),
    n_estres_moderado=("nivel_estres",lambda s:(s=="Moderado").sum()),
    n_estres_bajo=("nivel_estres",lambda s:(s=="Bajo").sum())).reset_index()
T2["pct_estres_alto"]=(T2["n_estres_alto"]/T2["n_total"]*100).round(2)
T2.to_sql("dwh_agg_prevalencia",engine,if_exists="append",index=False)
print(f"T2: {len(T2)} grupos → dwh_agg_prevalencia")
display(T2.sort_values("pct_estres_alto",ascending=False).head(8))

**Opinión.** La variabilidad por grupo demográfico es real: confirma que género y rango de edad añaden señal predictiva — justificando su inclusión como features en los modelos del Objetivo 2.

### T3 — Señales NLP de Dreaddit (resumen estadístico por grupo)

In [ ]:
tr=pd.read_csv(os.path.join(BASE,"Bloque 3","3 Stress Analysis in Social Media Dataset","dreaddit-train.csv"))
te=pd.read_csv(os.path.join(BASE,"Bloque 3","3 Stress Analysis in Social Media Dataset","dreaddit-test.csv"))
dr=pd.concat([tr,te],ignore_index=True)
dr=dr[~dr["text"].str.strip().isin(["#NAME?","#REF!","#VALUE!"])].copy()
dr["clase_estres"]=dr["label"].map({0:"No estres",1:"Estres"})
dr["n_words"]=dr["text"].str.split().str.len()
T3=dr.groupby(["subreddit","clase_estres"]).agg(
    n_posts=("text","size"),sentimiento_medio=("sentiment","mean"),
    liwc_ansiedad_medio=("lex_liwc_anx","mean"),liwc_emo_neg_medio=("lex_liwc_negemo","mean"),
    liwc_emo_pos_medio=("lex_liwc_posemo","mean"),longitud_media_palabras=("n_words","mean")
).round(4).reset_index()
T3.to_sql("dwh_nlp_senales",engine,if_exists="append",index=False)
print(f"T3: {len(T3)} combinaciones → dwh_nlp_senales")
display(T3.head(6))

**Opinión.** La señal LIWC es consistente en todos los subreddits: los posts con estrés siempre tienen mayor densidad de palabras de ansiedad. Esto avala la selección de features LIWC como predictores fuertes para el modelo NLP del Objetivo 2, antes de necesitar TF-IDF o embeddings.

### T4 — DASS por demografía con severidad clínica (pivote + clasificación por umbrales)

In [ ]:
d=pd.read_csv(os.path.join(BASE,"Bloque 4","4 Depression Anxiety Stress Scales Responses","data.csv"),sep="\t",low_memory=False)
d=d[(d["age"]>=18)&(d["age"]<=25)].copy()
depi=[3,5,10,13,16,17,21,24,26,31,34,37,38,42]; anxi=[2,4,7,9,15,19,20,23,25,28,30,36,40,41]; stri=[1,6,8,11,12,14,18,22,27,29,32,33,35,39]
d["dep"]=d[[f"Q{i}A" for i in depi]].sum(axis=1); d["anx"]=d[[f"Q{i}A" for i in anxi]].sum(axis=1); d["str"]=d[[f"Q{i}A" for i in stri]].sum(axis=1)
d["genero"]=d["gender"].map({1:"Masculino",2:"Femenino",3:"Otro"}).fillna("No reportado")
d["rango_edad"]="18-25"
d["nivel_educativo"]=d["education"].map({1:"Secundaria o menos",2:"Secundaria o menos",3:"Grado universitario",4:"Posgrado"}).fillna("No reportado")
T4=d.groupby(["genero","rango_edad","nivel_educativo"]).agg(
    n_respondentes=("dep","size"),depresion_media=("dep","mean"),ansiedad_media=("anx","mean"),
    estres_medio=("str","mean"),pct_estres_severo=("str",lambda s:(s>=40).mean()*100)
).round(2).reset_index()
T4.to_sql("dwh_dass_severidad",engine,if_exists="append",index=False)
print(f"T4: {len(T4)} grupos demográficos → dwh_dass_severidad")
display(T4.sort_values("estres_medio",ascending=False).head(6))

### T5 — Brecha de tratamiento (tabla de contingencia)

In [ ]:
mh=pd.read_csv(os.path.join(BASE,"Bloque 4","4 Mental Health Dataset","Mental Health dataset1.csv"))
mh=mh[mh["Occupation"]=="Student"].copy()
filas=[]
for dim,col in [("estres_creciente","IncreasingStress"),("antecedentes_familiares","FamilyHistory")]:
    for cat,sub in mh.groupby(col):
        filas.append({"dimension":dim,"categoria":str(cat),"n_total":len(sub),"n_con_tratamiento":int((sub["Treatment"]=="Yes").sum())})
T5=pd.DataFrame(filas); T5["pct_tratamiento"]=(T5["n_con_tratamiento"]/T5["n_total"]*100).round(2)
T5.to_sql("dwh_brecha_tratamiento",engine,if_exists="append",index=False)
print(f"T5: {len(T5)} filas → dwh_brecha_tratamiento")
display(T5)

---
## Cálculo de KPIs y almacenamiento con histórico

In [ ]:
kpis=[]
v=(T1["nivel_estres"]=="Alto").mean()*100
kpis.append((1,"KPI_ESTRES_ALTO",round(v,4),f"{(T1['nivel_estres']=='Alto').sum():,} de {len(T1):,}"))
amuj=d[d["genero"]=="Femenino"]["anx"].mean(); ahom=d[d["genero"]=="Masculino"]["anx"].mean()
kpis.append((2,"KPI_BRECHA_GENERO",round(amuj-ahom,4),f"Mujer {amuj:.1f} - Hombre {ahom:.1f}"))
dp=pd.read_csv(os.path.join(BASE,"Bloque 3","3 Student_Mental_Health_Analysis GITHUB","Depression+Student+Dataset.csv"))
ct=pd.crosstab(dp["Have you ever had suicidal thoughts ?"],dp["Depression"],normalize="index")*100
kpis.append((3,"KPI_RIESGO_CRITICO",round(ct.loc["Yes","Yes"],4),"% suicidas con depresion"))
kpis.append((4,"KPI_TASA_TRATAMIENTO",round((mh["Treatment"]=="Yes").mean()*100,4),f"{(mh['Treatment']=='Yes').sum():,} estudiantes"))
ae=dr[dr["clase_estres"]=="Estres"]["lex_liwc_anx"].mean(); an=dr[dr["clase_estres"]=="No estres"]["lex_liwc_anx"].mean()
kpis.append((5,"KPI_DISCRIMINANTE_LIWC",round(ae/an,4),f"estres {ae:.2f} / no-estres {an:.2f}"))
kpis.append((6,"KPI_COBERTURA_DEMO",round(T1["demografia_completa"].mean()*100,4),"% con genero+edad reportados"))
res=pd.DataFrame(kpis,columns=["id_kpi","codigo","valor","detalle"]); res["fecha_ejecucion"]=EJECUCION
res.to_sql("dwh_kpi_resultados",engine,if_exists="append",index=False)
print("KPIs almacenados:\n"); display(res[["codigo","valor","detalle"]])

---
## Verificación final

Lee los KPIs desde la base uniendo catálogo + último resultado.
Esta consulta es la que alimentará las tarjetas de KPI en Power BI (Objetivo 3).

In [ ]:
q = """
SELECT c.codigo, c.nombre, r.valor, c.unidad, r.detalle, r.fecha_ejecucion
FROM dwh_kpi_resultados r
JOIN dwh_kpi_catalogo c ON c.id_kpi = r.id_kpi
WHERE r.fecha_ejecucion = (SELECT MAX(fecha_ejecucion) FROM dwh_kpi_resultados)
ORDER BY c.id_kpi
"""
display(pd.read_sql(text(q), engine))
print("\n✅ ETL ENTREGA 2 COMPLETADO — Notebook 08 v1")
print("   Tablas generadas: dwh_hechos_estres_unificado, dwh_agg_prevalencia,")
print("   dwh_nlp_senales, dwh_dass_severidad, dwh_brecha_tratamiento,")
print("   dwh_kpi_catalogo, dwh_kpi_resultados")
